# Review-first LLM planning for a custom clinical OCRL task

Only local metadata, aggregate coverage, and recognized dictionary rows may be sent to the model. The response is a non-executable draft and cannot run until it passes strict validation and explicit approval.

In [ ]:
import os
from pathlib import Path

from ConMedRL.data import (
    GenericPlanner,
    LLMConfig,
    PreprocessConfig,
    approve_plan,
    build_dataset,
    get_llm_backend,
    profile_dataset,
)

DATA_DIR = Path(os.environ.get("CLINICAL_DATA_DIR", "../local_icu_data"))
TASK_DESCRIPTION = (
    "Optimize hourly vasopressor dose. Minimize 28-day mortality while "
    "constraining acute kidney injury. Use only pre-decision information."
)
RUN_LLM = False
APPROVE_AFTER_REVIEW = False

# profile_dataset keeps patient rows local. to_llm_payload() contains only
# filenames, headers, aggregate facts, and recognized dictionary rows.
if RUN_LLM:
    profile = profile_dataset(DATA_DIR)
    backend = get_llm_backend(LLMConfig(
        provider="openai",
        api_key=os.environ["OPENAI_API_KEY"],
    ))
    draft = GenericPlanner(backend).recommend_task_plan(profile, TASK_DESCRIPTION)
    print("Approved?", draft.is_approved)
    print("Confidence:", draft.confidence)
    print("Warnings:", list(draft.warnings))
    print("Unresolved:", list(draft.unresolved_decisions))

    # Inspect draft.to_dict() and verify clinical definitions, source columns,
    # units, time windows, leakage risk, actions, and all cost rules first.
    if APPROVE_AFTER_REVIEW:
        approved = approve_plan(draft, profile)
        config = PreprocessConfig(
            database="generic",
            task=approved.tasks[0].name,
            data_dir=DATA_DIR,
            output_dir="./processed_custom_task",
            dataset_spec=approved,
        )
        bundle = build_dataset(config)
        print(bundle.summary())

## Safety boundary

A configured provider is not permission to resolve variables during ordinary preprocessing. Plans are immutable data objects, every expression uses a fixed operator whitelist, selected names must exactly match profiler offerings, and source fingerprints participate in the approval hash. Low confidence, unresolved decisions, incompatible units, unsupported windows, possible leakage, stale files, or a changed plan block execution. Never paste an API key into a notebook; use an environment variable.